<div style="width: 76ch;">
<h3>Ruitewissers</h3>
(<em>Colin Cools, Lander Cortens</em>)



</div>

In [1]:
# Animation of four bar


import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

%matplotlib widget

In [ ]:
# kinematic parameters, in SI units:
r1   = 0.22 #m
r2   = 0.33 #m
r3   = 0.75 #m
r4   = 0.50 #m
r5   = 0.58 #m
r6   = 0.60 #m
r7   = 0.19 #m

rDC  = 0.08 #m

grondA_x = 0.00 #m
grondA_y = 0.00 #m

grondD_x = 0.35 #m
grondD_y = 0.30 #m

grondH_x = 0.85 #m
grondH_y = 0.00 #m

grondI_x = 0.85 #m
grondI_y = 0.30 #m


# angular position driver:
omega  = 0.5      # driver frequency [rad/s]
A      = 1        # amplitude []
theta1   = 1.22 + A * np.sin(omega * t)
dtheta1  = omega * A * np.cos(omega * t)
ddtheta2 = -omega ** 2 * A * np.sin(omega * t)

# configuration of numerical simulation interval and sampling:
t_begin =  0     # start time of simulation
t_end = np.pi/omega     # end time of simulation
Ts      =  0.01  # time step of simulation
t = np.arange(t_begin, t_end + Ts, Ts)  # time vector


# initial conditions for nonlinear solver ("fsolve") of position closure:
# phi1_init = 1.22 #rad volgens mij is deze bepaald door de motor, dus mag deze weg
theta2_init = 0.26 #rad
theta3_init = 2.09 #rad
theta4_init = 5.76 #rad
theta5_init = 5.41 #rad
theta6_init = 1.05 #rad
theta7_init = 1.05 #rad
# (different choices can lead to different topologies of your mechanism)

In [ ]:
# Function to rotate a vector z over an angle theta:
def rotate_vector(z, theta):
    rotation_matrix = np.array([[np.cos(theta), -np.sin(theta)],
                                [np.sin(theta), np.cos(theta)]])
    return np.dot(rotation_matrix, z)

In [ ]:
# Define function to compute "gap" in position closure:
def loop_closure_eqs(theta_init, theta1, r1, r2, r3, r4, r5, r6, r7, rDC, grondA_x, grondA_y, grondD_x, grondD_y, grondH_x, grondH_y, grondI_x, grondI_y):
    theta2 = theta_init[0]
    theta3 = theta_init[1]
    theta4 = theta_init[2]
    theta5 = theta_init[3]
    theta6 = theta_init[4]
    theta7 = theta_init[5]
    

    # Loop closure gaps:
    F1 = r1 * np.cos(theta1) + r2 * np.cos(theta2) + rDC * np.cos(theta3) - (grondD_x-grondA_x)
    F2 = r2 * np.sin(theta2) + r3 * np.sin(theta3) - r4 * np.sin(theta4) - r1 * np.sin(theta1)

    return [F1, F2]

In [ ]:
def kinematics_4bar(r1, r2, r3, r4, phi1, phi2, dphi2, ddphi2, phi3_init, phi4_init, t):
  optim_options = {"full_output":True}  # options for fsolve

  # numerically solving the position closure at all sampling times:
  for k, time in enumerate(t):
      # Position Analysis
      x, _, ier, message  = fsolve(lambda x: loop_closure_eqs(x, phi2[k], r1, r2, r3, r4, phi1), [phi3_init, phi4_init], **optim_options)

      if ier != 1:
          print("The fsolve exit flag was not 1, probably no convergence!")
          print(message)

      phi3[k] = x[0]
      phi4[k] = x[1]

      # velocity closure:
      A = np.array([[-r3 * np.sin(phi3[k]), r4 * np.sin(phi4[k])],
                    [r3 * np.cos(phi3[k]), -r4 * np.cos(phi4[k])]])
      B = np.array([r2 * np.sin(phi2[k]) * dphi2[k],
                    -r2 * np.cos(phi2[k]) * dphi2[k]])

      x = np.linalg.solve(A, B)
      dphi3[k] = x[0]
      dphi4[k] = x[1]
      cond[k]  = np.linalg.cond(A)

      # acceleration closure:
      B = np.array([r2 * np.cos(phi2[k]) * dphi2[k]**2 + r2 * np.sin(phi2[k]) * ddphi2[k] + r3 * np.cos(phi3[k]) * dphi3[k]**2 - r4 * np.cos(phi4[k]) * dphi4[k]**2,
                    r2 * np.sin(phi2[k]) * dphi2[k]**2 - r2 * np.cos(phi2[k]) * ddphi2[k] + r3 * np.sin(phi3[k]) * dphi3[k]**2 - r4 * np.sin(phi4[k]) * dphi4[k]**2])

      # Note: A matrix is the same as for velocities.
      x = np.linalg.solve(A, B)
      ddphi3[k] = x[0]
      ddphi4[k] = x[1]

      # Next iteration with initial values from by simple integration:
      phi3_init = phi3[k] + (t[1] - t[0]) * dphi3[k]
      phi4_init = phi4[k] + (t[1] - t[0]) * dphi4[k]

  return phi3, phi4, dphi3, dphi4, ddphi3, ddphi4, cond